In [ ]:
import dask.dataframe as dd
from dask.distributed import Client



<Client: 'tcp://127.0.0.1:50795' processes=4 threads=8, memory=7.75 GiB>


In [ ]:
client = Client(n_workers=4)  
print("Dask Client Started! Monitor at:", client.dashboard_link)

Data loading in chunks 


In [ ]:
csv_path = "path/to/soil_data.csv"  

print("Loading CSV in chunks...")
df = dd.read_csv(csv_path, dtype={
    "timestamp": "int64",
    "soil_moisture": "float32",
    "soil_water_content": "float32",
    "carbon_percent": "float32",
    "nitrogen_percent": "float32",
    "atmospheric_humidity": "float32",
    "temperature": "float32",
    "pH": "float32",
})

print("CSV Loaded Successfully! First 5 rows:")
print(df.head())  


✅ CSV Loaded Successfully!


converting it into parquet for better computation

In [ ]:
parquet_path = "path/to/soil_data.parquet"


print("Converting CSV to Parquet...")
df.to_parquet(parquet_path, engine="pyarrow", compression="snappy") 

print("CSV Converted to Parquet! Future runs will be 5-10x Faster.")


✅ CSV Converted to Parquet! Future runs will be 5-10x Faster.


In [ ]:
print("Loading optimized Parquet file...")
df = dd.read_parquet(parquet_path, engine="pyarrow")

print("Parquet Loaded! First 5 rows:")
print(df.head())


✅ Parquet Loaded Successfully!


Data Cleaning

In [ ]:
print("Cleaning data (removing NaNs, fixing invalid values)...")
df = df.dropna()  # Remove missing values
df = df[df["pH"].between(4, 9)]  # pH must be between 4-9
df = df[df["temperature"].between(0, 40)]  # Valid temperature range

print("Data Cleaned! Sample after cleaning:")
print(df.head())


✅ Data Cleaned!


Data sorting for some parameter 

In [ ]:
print("Sorting data by timestamp, soil moisture, and temperature...")
df = df.sort_values(["timestamp", "soil_moisture", "temperature"])

print("Filtering: Selecting rows where soil_moisture > 80 & pH < 5...")
filtered_df = df[(df["soil_moisture"] > 80) & (df["pH"] < 5)]

filtered_df.compute()  # Executes in parallel
print("Data Sorted & Filtered! Sample filtered rows:")
print(filtered_df.head())


Stastics analysis

In [ ]:
print("Computing descriptive statistics for carbon & nitrogen...")
stats = df[['carbon_percent', 'nitrogen_percent']].describe().compute()

print("Descriptive Statistics:\n", stats)


Interpolization

In [ ]:
print("Performing linear interpolation to fill missing values...")
df = df.interpolate(method="linear").compute()

print("Missing Data Interpolated! Sample after interpolation:")
print(df.head())


ML model 

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

print("Sampling 1% of data for fast ML training...")
df_sample = df.sample(frac=0.01).compute()

X = df_sample[['soil_moisture', 'temperature']]
y = df_sample['carbon_percent']

print("Splitting data into training and test sets...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training a Linear Regression Model...")
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("ML Model Training Complete! First 5 predictions:")
print(y_pred[:5])


Finally we are savin te data 

In [ ]:
processed_parquet_path = "path/to/processed_soil_data.parquet"

print("Saving cleaned & processed data for future use...")
df.to_parquet(processed_parquet_path, engine="pyarrow", compression="gzip")

print(" Processed Data Saved Successfully!")
